# Dear Canada — Place Mention Extraction Pipeline (experimental)

Extracts place mentions from every diary entry in the *Dear Canada* corpus, standardises entry dates, attempts to geocode each place, and writes one row per mention.

Status: **experiment**. Uses the general-purpose spaCy NER model without fine-tuning. Known concessions are listed under *Known limitations*.

## 0. Requirements, inputs and outputs

### Environment
- Python 3.10 or later, run locally in Jupyter.
- Packages: `pandas`, `spacy`, `requests`; `tqdm` optional (progress bars).
- spaCy model: `python -m spacy download en_core_web_sm`

### Data the notebook reads
| Item | Location | Format |
|---|---|---|
| Entry files | `ENTRIES_DIR` (set in the paths cell) | One file per book, named `<bookcode>_entries.csv`, UTF-8 |
| Columns | | `book, chapter_id, entry_id, date, entry_text` |

Assumptions: `entry_text` is cleaned but not flattened (capitalisation and punctuation retained). `(book, entry_id)` identifies an entry. `date` is free-form and is normalised in section 2 using the `DEFAULT_YEARS` table in the configuration cell. Books missing from that table receive no inferred year.

### Network access
Outbound HTTPS to `nominatim.openstreetmap.org` (section 4). The public service permits at most one request per second and requires an identifying `User-Agent`; set `NOMINATIM_USER_AGENT` in the configuration cell before running. All responses are cached on disk, so a run can be interrupted and resumed, and a re-run makes no network calls for places already queried.

### Disk access
Writes to `OUTPUT_DIR` only.

### Outputs
| File | Content |
|---|---|
| `place_mentions.csv` | Main result. One row per place mention. |
| `unique_places.csv` | One row per distinct place string, with mention count, NER labels and resolution outcome. Intended for review and as a source for training data. |
| `geocode_cache.sqlite` | Raw Nominatim responses. Delete to force re-querying. |

Schema of `place_mentions.csv`:

| Column | Type | Description |
|---|---|---|
| `book` | str | Book code |
| `chapter_id` | str | Chapter identifier from the source file |
| `entry_id` | str | Entry identifier from the source file |
| `date` | str | Entry date, ISO 8601, at the precision in `date_precision` (`YYYY-MM-DD`, `YYYY-MM`, `YYYY`) or null |
| `date_precision` | str | `day`, `month`, `year` or `unknown` |
| `year_inferred` | bool | True if the year was filled from `DEFAULT_YEARS` rather than present in the source date |
| `date_raw` | str | Original date string |
| `place` | str | Cleaned place string as mentioned in the text |
| `ner_label` | str | spaCy label: `GPE`, `LOC`, `FAC` or `ORG` |
| `char_start`, `char_end` | int | Character offsets of the mention in `entry_text` |
| `historicity` | bool | True if the place was matched to a real-world location (see section 5) |
| `resolution_status` | str | `override`, `resolved_ca`, `resolved_world`, `unresolved`, `lookup_error`, `skipped_label` or `not_attempted` |
| `region_historical` | str | Political unit at the entry date (e.g. Upper Canada). Null if not historical. |
| `region_modern` | str | Modern province, territory, state or country |
| `country` | str | Modern country |
| `latitude`, `longitude` | float | WGS84. Null unless `historicity` is True. |
| `matched_name` | str | Gazetteer name the place was matched to |

### Known limitations
- Story locations (e.g. "the Hall's farm") are generally not detected by the generic model; where detected, they are indistinguishable from failed geocoding except through `resolution_status`.
- `historicity` means "matched to a real-world gazetteer entry", not "verified historical place".
- Geocoding uses modern names and modern positions.
- The historical region table covers Canadian provinces and territories only and is a draft requiring historian review.

In [1]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Every file system location used by the notebook is defined here.
from pathlib import Path

PROJECT_ROOT       = Path(".").resolve()

# Inputs
ENTRIES_DIR        = PROJECT_ROOT / "data"   # folder holding *_entries.csv
ENTRIES_GLOB       = "*_entries.csv"

# Outputs
OUTPUT_DIR         = PROJECT_ROOT / "output"
MENTIONS_CSV       = OUTPUT_DIR / "place_mentions.csv"
UNIQUE_PLACES_CSV  = OUTPUT_DIR / "unique_places.csv"
GEOCODE_CACHE_PATH = OUTPUT_DIR / "geocode_cache.sqlite"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Entries folder :", ENTRIES_DIR, "(exists)" if ENTRIES_DIR.exists() else "(NOT FOUND)")
print("Output folder  :", OUTPUT_DIR)

Entries folder : /Users/sophiebird/Downloads/Dear Canada/NLP and Data Creation from Anthology/data (exists)
Output folder  : /Users/sophiebird/Downloads/Dear Canada/NLP and Data Creation from Anthology/output


In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────
# NER
SPACY_MODEL         = "en_core_web_sm"
PLACE_LABELS        = {"GPE", "LOC", "FAC", "ORG"}   # labels kept as place mentions
NER_BATCH_SIZE      = 64
BOOKS               = None        # None = all books; or a list of codes, e.g. ["AToBD", "DofmyC"]

# Geocoding
NOMINATIM_URL        = "https://nominatim.openstreetmap.org/search"
NOMINATIM_USER_AGENT = "DearCanada-place-pipeline/0.1 (contact: YOUR_EMAIL_HERE)"   # REQUIRED: edit
REQUEST_INTERVAL_S   = 1.1        # minimum seconds between uncached requests
PRIMARY_COUNTRY_CODE = "ca"       # searched first
FALLBACK_WORLDWIDE   = True       # search without country restriction if no Canadian match
MATCH_THRESHOLD      = 0.85       # minimum name similarity (0-1) to accept a gazetteer result
GEOCODE_LABELS       = {"GPE", "LOC", "FAC", "ORG"}   # labels for which geocoding is attempted
MAX_UNIQUE_TO_GEOCODE = None      # None = no cap; integer = geocode only the N most frequent places

# Known historical year per book, used to fill dates without a year.
# Add the remaining books here.
DEFAULT_YEARS = {
    "AinUTL": 1666, "AToBD": 1862, "BFfH": 1916, "BUoL": 1885,
    "CofOO": 1866, "DofmyC": 1759, "DofTT": 1887, "DRtoF": 1863,
    "EftW": 1940, "FitS": 1815, "NbutC": 1783, "NSH": 1917,
    "OA": 1922, "OatmyD": 1897, "PasWide": 1926, "PintPL": 1914,
    "SofS": 1847, "TA": 1941, "TFN": 1912, "WofW": 1812,
}

In [3]:
# ── Imports and dependency check ──────────────────────────────────────────────
import re, json, time, sqlite3, difflib, unicodedata
from datetime import datetime

import numpy as np
import pandas as pd
import requests
import spacy

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kwargs):
        return it

print("pandas", pd.__version__, "| spaCy", spacy.__version__, "| model", SPACY_MODEL)
if "YOUR_EMAIL" in NOMINATIM_USER_AGENT:
    print("WARNING: set NOMINATIM_USER_AGENT in the configuration cell before geocoding.")

pandas 3.0.0 | spaCy 3.8.16 | model en_core_web_sm


## 1. Load entries

Reads every file matching `ENTRIES_GLOB` in `ENTRIES_DIR` and concatenates them into one table. Required columns are validated per file. Where the `book` column is empty, the book code is taken from the filename.

In [4]:
REQUIRED_COLUMNS = ["book", "chapter_id", "entry_id", "date", "entry_text"]

def load_entries(entries_dir, pattern, books=None):
    files = sorted(Path(entries_dir).glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files matching {pattern} in {entries_dir}")
    frames = []
    for f in files:
        code_from_name = f.name[: -len("_entries.csv")]
        if books is not None and code_from_name not in books:
            continue
        df = pd.read_csv(f, dtype=str, keep_default_na=False, encoding="utf-8")
        missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
        if missing:
            raise ValueError(f"{f.name}: missing columns {missing}")
        df["book"] = df["book"].replace("", code_from_name)
        mismatched = set(df["book"]) - {code_from_name}
        if mismatched:
            print(f"NOTE {f.name}: 'book' column contains {mismatched}, differs from filename code")
        frames.append(df[REQUIRED_COLUMNS])
    if not frames:
        raise ValueError("No books selected; check BOOKS in the configuration cell.")
    return pd.concat(frames, ignore_index=True)

entries = load_entries(ENTRIES_DIR, ENTRIES_GLOB, BOOKS).rename(columns={"date": "date_raw"})

dupes = entries.duplicated(["book", "entry_id"], keep=False).sum()
if dupes:
    print(f"WARNING: {dupes} rows share a (book, entry_id) with another row.")

print(f"{entries['book'].nunique()} books, {len(entries)} entries")
entries.groupby("book").size().rename("entries").to_frame().T

20 books, 3266 entries


book,AToBD,AinUTL,BFfH,BUoL,CofOO,DRtoF,DofTT,DofmyC,EftW,FitS,NSH,NbutC,OA,OatmyD,PasWide,PintPL,SofS,TA,TFN,WofW
entries,114,167,134,129,258,201,169,175,259,142,126,155,123,71,193,230,137,274,70,139


## 2. Date standardisation

`normalise_date` is the existing normalisation routine, applied unchanged. Its output (`Month DD, YYYY`) is then converted to ISO 8601 by `to_iso`, with a precision flag.

In [5]:
# ── Date normalisation (existing routine) ─────────────────────────────────────
FR_MONTHS = {
    "janvier": "January", "février": "February", "fevrier": "February",
    "mars": "March", "avril": "April", "mai": "May", "juin": "June",
    "juillet": "July", "août": "August", "aout": "August",
    "septembre": "September", "octobre": "October",
    "novembre": "November", "décembre": "December", "decembre": "December"
}
EN_MONTHS = [
    "January","February","March","April","May","June",
    "July","August","September","October","November","December"
]

def normalise_date(raw: str, default_year: int | None = None) -> str:
    if not isinstance(raw, str):
        return "Unknown Date"
    s = raw.strip()

    fr = re.search(r"le\s+(\d{1,2})\s+(" + "|".join(FR_MONTHS) + r")\s+(\d{4})", s, re.I)
    if fr:
        return f"{FR_MONTHS[fr.group(2).lower()]} {fr.group(1).zfill(2)}, {fr.group(3)}"

    dmy = re.search(r"(\d{1,2})\s+(" + "|".join(EN_MONTHS) + r")\s+(\d{4})", s, re.I)
    if dmy:
        return f"{dmy.group(2).capitalize()} {dmy.group(1).zfill(2)}, {dmy.group(3)}"

    mdy = re.search(r"(" + "|".join(EN_MONTHS) + r")\s+(\d{1,2})(?:st|nd|rd|th)?[,\s]+(\d{4})", s, re.I)
    if mdy:
        return f"{mdy.group(1).capitalize()} {mdy.group(2).zfill(2)}, {mdy.group(3)}"

    wd = re.search(r"(?:Mon|Tue|Wed|Thu|Fri|Sat|Sun)[a-z]*,?\s*(" + "|".join(EN_MONTHS) + r")\s+(\d{1,2})", s, re.I)
    if wd:
        m, d = wd.group(1).capitalize(), wd.group(2).zfill(2)
        return f"{m} {d}, {default_year}" if default_year else f"{m} {d}"

    md = re.search(r"(" + "|".join(EN_MONTHS) + r")\s+(\d{1,2})\b", s, re.I)
    if md:
        m, d = md.group(1).capitalize(), md.group(2).zfill(2)
        return f"{m} {d}, {default_year}" if default_year else f"{m} {d}"

    my = re.search(r"(" + "|".join(EN_MONTHS) + r")\s+(\d{4})", s, re.I)
    if my:
        return f"{my.group(1).capitalize()} {my.group(2)}"

    yr = re.search(r"\b(1[5-9]\d{2})\b", s)
    if yr:
        return yr.group(1)

    return s

> **[DESIGN NOTE — delete before publishing]**
> Changes made around the existing routine:
> 1. **Missing default years.** `Series.map` yields `NaN` for books absent from `DEFAULT_YEARS`, and `NaN` is truthy in Python, so `normalise_date` would have produced strings like `"March 12, nan"`. `NaN` is converted to `None` before the call. Books without a default year are reported.
> 2. **Final "fill missing year" step replaced.** The original appended `", <year>"` to any string without a four-digit year, including non-dates such as `"Spring"` or `"Unknown Date"`, producing unparseable strings. `to_iso` performs the same fallback (use the book's default year) but records the result as year-precision.
> 3. **ISO 8601 with a precision flag.** ISO strings sort correctly and parse in every tool downstream (pandas, KG loaders). Partial dates stay partial (`1862-05`, `1862`) rather than being padded to a false day. `year_inferred` marks values whose year came from `DEFAULT_YEARS`, so inferred years can be filtered out of any date-sensitive analysis.
> 4. The original `Month DD, YYYY` form is not kept in the output; `date_raw` retains the source string.

In [6]:
def to_iso(normalised: str, default_year: int | None):
    """Convert normalise_date output to (iso_string, precision)."""
    s = str(normalised).strip()
    for fmt, precision, out_fmt in (("%B %d, %Y", "day", "%Y-%m-%d"),
                                    ("%B %Y",     "month", "%Y-%m")):
        try:
            return datetime.strptime(s, fmt).strftime(out_fmt), precision
        except ValueError:
            pass
    if re.fullmatch(r"\d{4}", s):
        return s, "year"
    m = re.search(r"\b(1[5-9]\d{2})\b", s)
    if m:
        return m.group(1), "year"
    if default_year:
        return str(default_year), "year"
    return None, "unknown"

entries["default_year"] = entries["book"].map(DEFAULT_YEARS)

no_default = sorted(entries.loc[entries["default_year"].isna(), "book"].unique())
if no_default:
    print(f"NOTE: no DEFAULT_YEARS entry for {len(no_default)} book(s): {no_default}")

def _standardise(row):
    dy = None if pd.isna(row["default_year"]) else int(row["default_year"])
    normalised = normalise_date(row["date_raw"], dy)
    iso, precision = to_iso(normalised, dy)
    return pd.Series({"date": iso, "date_precision": precision})

entries[["date", "date_precision"]] = entries.apply(_standardise, axis=1)
entries["year_inferred"] = (
    entries["date"].notna() & ~entries["date_raw"].fillna("").str.contains(r"\b\d{4}\b")
)

print(entries["date_precision"].value_counts().to_string())
print()
entries.groupby("book").head(1)[["book", "date_raw", "date", "date_precision", "year_inferred"]].to_string(index=False)

date_precision
day      3169
month      50
year       47



'   book                      date_raw       date date_precision  year_inferred\n  AToBD                  May 15, 1862 1862-05-15            day          False\n AinUTL                 le 8 mai 1666 1666-05-08            day          False\n   BFfH             December 25, 1916 1916-12-25            day          False\n   BUoL           Le 31 décembre 1884 1884-12-31            day          False\n  CofOO                    April 1866    1866-04          month          False\n  DRtoF                  January 1863    1863-01          month          False\n  DofTT                  May 19, 1887 1887-05-19            day          False\n DofmyC               Le 8 avril 1759 1759-04-08            day          False\n   EftW         Monday, June 17, 1940 1940-06-17            day          False\n   FitS                       July 15 1815-07-15            day           True\n    NSH Wednesday, September 26, 1917 1917-09-26            day          False\n  NbutC             October 6th, 1783 1

## 3. Named entity recognition

Runs the pre-trained spaCy pipeline over every `entry_text` and keeps entities whose label is in `PLACE_LABELS`. Each entity becomes one mention row with character offsets relative to `entry_text`. Mention strings are cleaned by collapsing whitespace, straightening apostrophes, removing a leading article and trimming surrounding punctuation.

> **[DESIGN NOTE — delete before publishing]**
> - **Pipeline components disabled** (`lemmatizer`, `parser`, `tagger`, `attribute_ruler`): the NER component in `en_core_web_sm` does not depend on them, and disabling them speeds up a full-corpus pass. If a different model is substituted in `SPACY_MODEL`, re-check this; transformer models share an encoder with other components.
> - **`ORG` retained** as instructed. It adds noise (people, companies, institutions) and is the main source of false geocoding candidates; the name-similarity threshold in section 4 filters most of these. Exclude it via `PLACE_LABELS` or `GEOCODE_LABELS`.
> - **Leading articles removed** so "the Citadel" and "Citadel" share a lookup. Possessive `'s` is **not** stripped: it would corrupt names such as "St. John's".
> - **Plausibility filter**: strings under three characters or with no letters are dropped.

In [7]:
nlp = spacy.load(SPACY_MODEL, disable=["lemmatizer", "parser", "tagger", "attribute_ruler"])

_LEADING_ARTICLE = re.compile(r"^(?:the|a|an)\s+", re.I)

def clean_mention(text: str) -> str:
    t = " ".join(text.split()).replace("’", "'").replace("‘", "'")
    t = _LEADING_ARTICLE.sub("", t)
    return t.strip(" ,.;:\"'“”()")

def is_plausible(place: str) -> bool:
    return len(place) >= 3 and any(c.isalpha() for c in place)

texts = entries["entry_text"].fillna("").tolist()
rows = []
for idx, doc in zip(entries.index, tqdm(nlp.pipe(texts, batch_size=NER_BATCH_SIZE), total=len(texts), desc="NER")):
    for ent in doc.ents:
        if ent.label_ not in PLACE_LABELS:
            continue
        place = clean_mention(ent.text)
        if not is_plausible(place):
            continue
        rows.append({"entry_idx": idx, "place": place, "ner_label": ent.label_,
                     "char_start": ent.start_char, "char_end": ent.end_char})

mentions = pd.DataFrame(rows, columns=["entry_idx", "place", "ner_label", "char_start", "char_end"])
print(f"{len(mentions)} mentions in {mentions['entry_idx'].nunique()} of {len(entries)} entries")
print(mentions["ner_label"].value_counts().to_string())

/Users/sophiebird/anaconda3/envs/dearcanadaenv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


NER:   0%|          | 0/3266 [00:00<?, ?it/s]

9460 mentions in 2353 of 3266 entries
ner_label
ORG    4614
GPE    3813
LOC     524
FAC     509


## 4. Geocoding

Distinct place strings are deduplicated (case-, accent-, hyphen- and "Saint/St." -insensitive) and each is resolved once. Resolution order:

1. **Override table** (`PLACE_OVERRIDES`): manual entries for names the gazetteer cannot resolve correctly. Returns a representative point, not a boundary.
2. **Nominatim, restricted to `PRIMARY_COUNTRY_CODE`.**
3. **Nominatim, worldwide**, if `FALLBACK_WORLDWIDE` is set.

A Nominatim result is accepted only if its `name` has similarity ≥ `MATCH_THRESHOLD` to the query. The first accepted result in Nominatim's relevance order is used. Responses are cached in SQLite and keyed by query and country restriction.

Resolution statuses: `override`, `resolved_ca`, `resolved_world`, `unresolved` (queried, no acceptable match), `lookup_error` (network or service failure; not cached, retried on next run), `skipped_label` (no mention carries a label in `GEOCODE_LABELS`), `not_attempted` (excluded by `MAX_UNIQUE_TO_GEOCODE`).

> **[DESIGN NOTE — delete before publishing]**
> **Why Nominatim and not a gazetteer, and where a gazetteer fits.** A gazetteer (the Canadian Geographical Names Database, GeoNames, or a historical gazetteer) is the better long-term component for this step: it is offline, has no rate limit, covers small Canadian settlements more completely, and lets you disambiguate against a controlled list. Its costs are a download, a schema to learn, and still-modern names. For this experiment Nominatim needs no setup. All lookup logic is isolated in `resolve_place()`, so a gazetteer can replace `nominatim_search()` without changing the rest of the notebook. `PLACE_OVERRIDES` is a minimal hand-built gazetteer for historical names that no modern source resolves.
>
> **Why name-similarity acceptance instead of taking the top result.** Nominatim always returns something if anything loosely matches. Without a check, strings like "Company" or "Parliament" resolve to arbitrary locations and `historicity` becomes meaningless. Comparing the query to the result's `name` rejects these. Cost: legitimate places referred to by a descriptive phrase ("the Parliament building") will fail. This is a precision-over-recall choice; lower `MATCH_THRESHOLD` to reverse it.
>
> **Why Canada first, then worldwide.** Ambiguous names ("London", "York", "Victoria") resolve to the Canadian instance whenever one exists. A diary set in Britain that mentions London will resolve to London, Ontario. Fixing this needs per-book country context (a later metadata column).
>
> **Why "Quebec" is an override.** Nominatim ranks the province above the city for "Quebec"; diary usage almost always means the city. Add further overrides of this kind as the review file (`unique_places.csv`) reveals them.
>
> **Why SQLite for the cache.** Safe against interruption (each response is committed on receipt), no extra dependency, and queryable.

In [8]:
# ── Name folding ──────────────────────────────────────────────────────────────
def fold(s: str) -> str:
    """Normalise a place string for comparison and deduplication."""
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c)).lower()
    s = s.replace("'", "").replace(".", "")
    s = re.sub(r"[-‐‑–—/]", " ", s)
    s = re.sub(r"\bsaint\b", "st", s)
    s = re.sub(r"\bsainte\b", "ste", s)
    return " ".join(s.split())

def name_similarity(a: str, b: str) -> float:
    return difflib.SequenceMatcher(None, fold(a), fold(b)).ratio()

In [9]:
# ── Override table ────────────────────────────────────────────────────────────
# Keys are fold()ed place strings. Coordinates are representative points
# (capital or principal settlement), NOT centroids or boundaries.
PLACE_OVERRIDES = {
    "quebec":       dict(matched_name="Quebec City (override)", latitude=46.8139, longitude=-71.2080,
                         region_modern="Quebec", country="Canada", country_code="ca"),
    "new france":   dict(matched_name="New France, represented by Quebec City (override)", latitude=46.8139, longitude=-71.2080,
                         region_modern="Quebec", country="Canada", country_code="ca"),
    "lower canada": dict(matched_name="Lower Canada, represented by Quebec City (override)", latitude=46.8139, longitude=-71.2080,
                         region_modern="Quebec", country="Canada", country_code="ca"),
    "upper canada": dict(matched_name="Upper Canada, represented by York/Toronto (override)", latitude=43.6532, longitude=-79.3832,
                         region_modern="Ontario", country="Canada", country_code="ca"),
    "acadia":       dict(matched_name="Acadia, represented by Annapolis Royal (override)", latitude=44.7431, longitude=-65.5187,
                         region_modern="Nova Scotia", country="Canada", country_code="ca"),
    "rupert's land": dict(matched_name="Rupert's Land, represented by York Factory (override)", latitude=57.0, longitude=-92.3,
                         region_modern="Manitoba", country="Canada", country_code="ca"),
}
PLACE_OVERRIDES = {fold(k): v for k, v in PLACE_OVERRIDES.items()}

In [10]:
# ── Cache and Nominatim client ────────────────────────────────────────────────
class GeocodeCache:
    def __init__(self, path):
        self.conn = sqlite3.connect(str(path))
        self.conn.execute("CREATE TABLE IF NOT EXISTS responses (key TEXT PRIMARY KEY, response TEXT)")
        self.conn.commit()

    def get(self, key):
        row = self.conn.execute("SELECT response FROM responses WHERE key = ?", (key,)).fetchone()
        return json.loads(row[0]) if row else None

    def put(self, key, value):
        self.conn.execute("INSERT OR REPLACE INTO responses (key, response) VALUES (?, ?)",
                          (key, json.dumps(value)))
        self.conn.commit()

cache = GeocodeCache(GEOCODE_CACHE_PATH)
_last_request_time = 0.0

def cache_key(query, country_code):
    return f"{query}|{country_code or '*'}"

def nominatim_search(query, country_code=None, retries=3):
    """Return the list of Nominatim results (possibly empty), or None on failure."""
    global _last_request_time
    key = cache_key(query, country_code)
    cached = cache.get(key)
    if cached is not None:
        return cached

    params = {"q": query, "format": "jsonv2", "addressdetails": 1,
              "limit": 5, "accept-language": "en"}
    if country_code:
        params["countrycodes"] = country_code

    for attempt in range(retries):
        wait = REQUEST_INTERVAL_S - (time.time() - _last_request_time)
        if wait > 0:
            time.sleep(wait)
        _last_request_time = time.time()
        try:
            resp = requests.get(NOMINATIM_URL, params=params,
                                headers={"User-Agent": NOMINATIM_USER_AGENT}, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            cache.put(key, data)
            return data
        except (requests.RequestException, ValueError):
            time.sleep(2 ** attempt)
    return None

In [11]:
# ── Resolution ────────────────────────────────────────────────────────────────
RESOLUTION_FIELDS = ["resolution_status", "matched_name", "latitude", "longitude",
                     "region_modern", "country", "country_code", "name_similarity"]

def _empty(status):
    return {**{f: None for f in RESOLUTION_FIELDS}, "resolution_status": status}

def pick_match(query, results):
    """First result, in Nominatim's order, whose name is similar enough to the query."""
    for r in results:
        sim = name_similarity(query, r.get("name", ""))
        if sim >= MATCH_THRESHOLD:
            return r, sim
    return None, 0.0

def resolve_place(place):
    key = fold(place)
    if key in PLACE_OVERRIDES:
        return {**_empty("override"), **PLACE_OVERRIDES[key], "name_similarity": 1.0}

    attempts = [(PRIMARY_COUNTRY_CODE, "resolved_ca")]
    if FALLBACK_WORLDWIDE:
        attempts.append((None, "resolved_world"))

    for country_code, status in attempts:
        results = nominatim_search(place, country_code)
        if results is None:
            return _empty("lookup_error")
        match, sim = pick_match(place, results)
        if match:
            addr = match.get("address", {})
            country = addr.get("country")
            state = addr.get("state") or addr.get("province") or addr.get("region")
            return {
                "resolution_status": status,
                "matched_name": match.get("display_name"),
                "latitude": float(match["lat"]),
                "longitude": float(match["lon"]),
                "region_modern": (state or country or "").split(" / ")[0] or None,
                "country": country,
                "country_code": addr.get("country_code"),
                "name_similarity": round(sim, 3),
            }
    return _empty("unresolved")

In [12]:
# ── Build the table of distinct places and resolve each ───────────────────────
mentions["place_key"] = mentions["place"].map(fold)

unique = (mentions.groupby("place_key")
          .agg(place_query=("place", lambda s: s.value_counts().index[0]),
               n_mentions=("place", "size"),
               labels=("ner_label", lambda s: ",".join(sorted(set(s)))))
          .reset_index()
          .sort_values("n_mentions", ascending=False, kind="stable")
          .reset_index(drop=True))

unique["label_eligible"] = unique["labels"].map(lambda l: bool(set(l.split(",")) & GEOCODE_LABELS))
unique["rank_eligible"]  = True if MAX_UNIQUE_TO_GEOCODE is None else unique.index < MAX_UNIQUE_TO_GEOCODE

def _will_query(row):
    return row.label_eligible and row.rank_eligible

pending = [r for r in unique.itertuples()
           if _will_query(r) and r.place_key not in PLACE_OVERRIDES
           and cache.get(cache_key(r.place_query, PRIMARY_COUNTRY_CODE)) is None]
max_requests = len(pending) * (2 if FALLBACK_WORLDWIDE else 1)
print(f"{len(unique)} distinct places; {sum(_will_query(r) for r in unique.itertuples())} eligible for geocoding.")
print(f"{len(pending)} not yet cached: up to {max_requests} requests, "
      f"about {max_requests * REQUEST_INTERVAL_S / 60:.0f} min at {REQUEST_INTERVAL_S}s/request.")

2127 distinct places; 2127 eligible for geocoding.
2123 not yet cached: up to 4246 requests, about 78 min at 1.1s/request.


In [13]:
resolutions = []
for r in tqdm(list(unique.itertuples()), desc="Geocoding"):
    if not r.label_eligible:
        res = _empty("skipped_label")
    elif not r.rank_eligible:
        res = _empty("not_attempted")
    else:
        res = resolve_place(r.place_query)
    resolutions.append({"place_key": r.place_key, **res})

unique = unique.merge(pd.DataFrame(resolutions), on="place_key", how="left")
print(unique["resolution_status"].value_counts().to_string())

Geocoding:   0%|          | 0/2127 [00:00<?, ?it/s]

resolution_status
unresolved        869
resolved_ca       676
resolved_world    578
override            4


## 5. Historicity and region

**Historicity.** `historicity = True` when `resolution_status` is `override`, `resolved_ca` or `resolved_world`; otherwise `False`. Coordinates are retained only when `historicity` is True.

**Region.** `region_modern` is the province, territory or state returned by the gazetteer (country if none). `region_historical` is the political unit at the entry's year, looked up in `REGION_RULES`. It is null if the place is unresolved, falls back to `region_modern` if the entry year is unknown or no rule covers it, and equals `region_modern` for places outside Canada.

`REGION_RULES` maps each modern Canadian region to a list of `(first_year, last_year, historical_name)`; `None` is open-ended.

> **[DESIGN NOTE — delete before publishing]**
> **Historicity definition.** The requested definition ("real place" vs "story location") cannot be determined by the generic model. The operational definition above treats any gazetteer match as real. It produces false positives (a fictional "Fairview" that happens to exist in the gazetteer) and conflates story locations with failed lookups of real but obscure places. `resolution_status` keeps the two separable; replace this rule once hand-labelled training data exists.
>
> **Region table.** Boundaries are drawn from modern provinces, so a place is assigned the unit that governed its *modern* province's territory in that year. This is coarse (e.g. northern Ontario was Rupert's Land until 1912; Labrador's status is omitted; Cape Breton's separate administration is omitted; Vancouver Island and mainland British Columbia are merged). Year boundaries use treaty/act years (1763, 1791, 1841, 1867, etc.). The table is a draft for historian review, kept as plain data so it can be edited without touching the logic.

In [14]:
# ── Historical region rules ───────────────────────────────────────────────────
# Keys: fold()ed modern region names. Values: (first_year, last_year, historical_name), inclusive.
_RUPERT = "Rupert's Land / North-Western Territory"

REGION_RULES = {
    "quebec": [
        (None, 1762, "New France"),
        (1763, 1790, "Province of Quebec"),
        (1791, 1840, "Lower Canada"),
        (1841, 1866, "Province of Canada (Canada East)"),
        (1867, None, "Quebec"),
    ],
    "ontario": [
        (None, 1762, "New France"),
        (1763, 1790, "Province of Quebec"),
        (1791, 1840, "Upper Canada"),
        (1841, 1866, "Province of Canada (Canada West)"),
        (1867, None, "Ontario"),
    ],
    "nova scotia": [
        (None, 1712, "Acadia"),
        (1713, None, "Nova Scotia"),
    ],
    "new brunswick": [
        (None, 1712, "Acadia"),
        (1713, 1783, "Nova Scotia"),
        (1784, None, "New Brunswick"),
    ],
    "prince edward island": [
        (None, 1758, "Île Saint-Jean (New France)"),
        (1759, 1768, "Nova Scotia (St. John's Island)"),
        (1769, 1872, "Colony of Prince Edward Island"),
        (1873, None, "Prince Edward Island"),
    ],
    "newfoundland and labrador": [
        (None, 1948, "Newfoundland"),
        (1949, None, "Newfoundland and Labrador"),
    ],
    "manitoba": [
        (None, 1869, "Rupert's Land"),
        (1870, None, "Manitoba"),
    ],
    "saskatchewan": [
        (None, 1869, _RUPERT),
        (1870, 1904, "North-West Territories"),
        (1905, None, "Saskatchewan"),
    ],
    "alberta": [
        (None, 1869, _RUPERT),
        (1870, 1904, "North-West Territories"),
        (1905, None, "Alberta"),
    ],
    "british columbia": [
        (None, 1857, "Columbia District / New Caledonia"),
        (1858, 1870, "Colony of British Columbia / Vancouver Island"),
        (1871, None, "British Columbia"),
    ],
    "yukon": [
        (None, 1869, _RUPERT),
        (1870, 1897, "North-West Territories"),
        (1898, None, "Yukon"),
    ],
    "northwest territories": [
        (None, 1869, _RUPERT),
        (1870, None, "North-West Territories"),
    ],
    "nunavut": [
        (None, 1869, _RUPERT),
        (1870, 1998, "North-West Territories"),
        (1999, None, "Nunavut"),
    ],
}

def historical_region(region_modern, country_code, year):
    if not region_modern:
        return None
    if country_code != "ca" or year is None:
        return region_modern
    for first, last, name in REGION_RULES.get(fold(region_modern), []):
        if (first is None or year >= first) and (last is None or year <= last):
            return name
    return region_modern

## 6. Assemble and write outputs

In [15]:
RESOLVED = {"override", "resolved_ca", "resolved_world"}

result = (mentions
          .merge(entries[["book", "chapter_id", "entry_id", "date", "date_precision",
                          "year_inferred", "date_raw"]],
                 left_on="entry_idx", right_index=True, how="left")
          .merge(unique[["place_key"] + RESOLUTION_FIELDS], on="place_key", how="left"))

result["historicity"] = result["resolution_status"].isin(RESOLVED)

# Coordinates only where historicity is True
for col in ("latitude", "longitude"):
    result.loc[~result["historicity"], col] = np.nan
result.loc[~result["historicity"], ["matched_name", "region_modern", "country"]] = None

_year = result["date"].map(lambda d: int(d[:4]) if isinstance(d, str) and d[:4].isdigit() else None)
result["region_historical"] = [
    historical_region(rm, cc, y) if h else None
    for rm, cc, y, h in zip(result["region_modern"], result["country_code"], _year, result["historicity"])
]

OUTPUT_COLUMNS = ["book", "chapter_id", "entry_id", "date", "date_precision", "year_inferred", "date_raw",
                  "place", "ner_label", "char_start", "char_end",
                  "historicity", "resolution_status",
                  "region_historical", "region_modern", "country",
                  "latitude", "longitude", "matched_name"]

# Keep source order (entry order within each file, then position within the entry)
result = (result.sort_values(["entry_idx", "char_start"], kind="stable")[OUTPUT_COLUMNS]
          .reset_index(drop=True))

result.to_csv(MENTIONS_CSV, index=False, encoding="utf-8")
unique.drop(columns=["label_eligible", "rank_eligible"]).to_csv(UNIQUE_PLACES_CSV, index=False, encoding="utf-8")

print(f"Wrote {len(result)} rows  -> {MENTIONS_CSV}")
print(f"Wrote {len(unique)} rows  -> {UNIQUE_PLACES_CSV}")
result.head(10)

Wrote 9460 rows  -> /Users/sophiebird/Downloads/Dear Canada/NLP and Data Creation from Anthology/output/place_mentions.csv
Wrote 2127 rows  -> /Users/sophiebird/Downloads/Dear Canada/NLP and Data Creation from Anthology/output/unique_places.csv


,book,chapter_id,entry_id,date,date_precision,year_inferred,date_raw,place,ner_label,char_start,char_end,historicity,resolution_status,region_historical,region_modern,country,latitude,longitude,matched_name
0,AToBD,01,ch01_e2,1862-05-15,day,False,"May 15, 1862",Fort Garry,GPE,794,804,True,resolved_ca,Rupert's Land,Manitoba,Canada,49.833355,-97.161074,"Fort Garry, Winnipeg, Manitoba, R3T 0S4, Canada"
1,AToBD,01,ch01_e3,1862-05-17,day,False,"May 17, 1862",Canada,GPE,328,334,True,resolved_ca,Canada,Canada,Canada,61.066692,-107.991707,Canada
2,AToBD,01,ch01_e3,1862-05-17,day,False,"May 17, 1862",Red River Settlement,ORG,343,367,False,unresolved,NaN,NaN,NaN,NaN,NaN,NaN
3,AToBD,01,ch01_e3,1862-05-17,day,False,"May 17, 1862",British Columbia,GPE,439,455,True,resolved_ca,Colony of British Columbia / Vancouver Island,British Columbia,Canada,55.001251,-125.002441,"British Columbia, Canada"
4,AToBD,01,ch01_e4,1862-05-18,day,False,"May 18, 1862",Hudson Bay Company's,ORG,395,419,False,unresolved,NaN,NaN,NaN,NaN,NaN,NaN
5,AToBD,01,ch01_e5,1862-05-20,day,False,"May 20, 1862",Luella,ORG,518,524,True,resolved_world,Texas,Texas,United States,33.569552,-96.551100,"Luella, Grayson County, Texas, United States"
6,AToBD,01,ch01_e6,1862-05-22,day,False,"May 22, 1862",Toronto,GPE,81,88,True,resolved_ca,Province of Canada (Canada West),Ontario,Canada,43.653482,-79.383935,"Toronto, Ontario, Canada"
7,AToBD,01,ch01_e8,1862-05-25,day,False,"May 25, 1862",Fort Edmonton,FAC,115,128,True,resolved_ca,Rupert's Land / North-Western Territory,Alberta,Canada,53.498881,-113.586280,"Fort Edmonton, Whitemud Road NW, River Valley ..."
8,AToBD,01,ch01_e10,1862-05-27,day,False,"May 27, 1862",Canada West,GPE,1648,1659,False,unresolved,NaN,NaN,NaN,NaN,NaN,NaN
9,AToBD,01,ch01_e11,1862-05-29,day,False,"May 29, 1862",International,ORG,199,212,True,resolved_world,South Moravian Region,South Moravian Region,Czechia,49.195049,16.605579,"International, 200/16, Husova, Brno-město, Brn..."


## 7. Diagnostics

Summary statistics for judging whether the approach has merit. The review tables are the primary input for deciding what to annotate for fine-tuning.

In [16]:
print("Mentions by NER label and resolution status")
print(pd.crosstab(result["ner_label"], result["resolution_status"], margins=True).to_string())

print("\nShare of mentions with historicity = True, by book")
print(result.groupby("book")["historicity"].agg(mentions="size", resolved="mean").round(2).to_string())

Mentions by NER label and resolution status
resolution_status  override  resolved_ca  resolved_world  unresolved   All
ner_label                                                                 
FAC                       0          296              74         139   509
GPE                      58         1751            1606         398  3813
LOC                       5          326              94          99   524
ORG                      91         1592            1212        1719  4614
All                     154         3965            2986        2355  9460

Share of mentions with historicity = True, by book
         mentions  resolved
book                       
AToBD         146      0.73
AinUTL        574      0.78
BFfH          479      0.90
BUoL          648      0.72
CofOO         596      0.55
DRtoF         364      0.87
DofTT         548      0.56
DofmyC        344      0.76
EftW          578      0.81
FitS          349      0.58
NSH           484      0.71
NbutC         2

In [17]:
print("Most frequent resolved places")
print(unique[unique["resolution_status"].isin(RESOLVED)]
      [["place_query", "labels", "n_mentions", "resolution_status", "matched_name"]]
      .head(25).to_string(index=False))

Most frequent resolved places
place_query          labels  n_mentions resolution_status                                                                                                                                                 matched_name
     Canada             GPE         361       resolved_ca                                                                                                                                                       Canada
       Papa             ORG         268       resolved_ca                                                             Papa, Area B (Halfmoon Bay), Sunshine Coast Regional District, British Columbia, V0N 1Y2, Canada
    England             GPE         181    resolved_world                                                                                                                                      England, United Kingdom
     Jasper         GPE,ORG         122       resolved_ca                                                     

In [18]:
print("Most frequent unresolved places (candidates for story locations, overrides or NER errors)")
print(unique[unique["resolution_status"] == "unresolved"]
      [["place_query", "labels", "n_mentions"]]
      .head(25).to_string(index=False))

Most frequent unresolved places (candidates for story locations, overrides or NER errors)
        place_query  labels  n_mentions
 Dear Papa and Mama     ORG         166
         White Loon FAC,ORG          71
           Moushoom     ORG          66
     Dear Constance     ORG          60
             Kirsty     ORG          60
             Briney GPE,ORG          56
     Madame Laurent     ORG          45
             Laddie     ORG          41
            Britain     GPE          35
                Mam     ORG          31
            Wigwedi GPE,ORG          29
Sister Marie-France     ORG          26
              Lizby GPE,ORG          24
      Confederation     ORG          23
        Dear Canada     ORG          22
           Barnardo GPE,ORG          21
 North West Company     ORG          21
        Québec City     GPE          19
    Monsieur Dumont FAC,GPE          18
                B.C     GPE          17
         Commandant     ORG          17
            America     GPE   

In [19]:
print("Random sample of resolved mentions for manual spot-checking")
result[result["historicity"]].sample(min(20, int(result["historicity"].sum())), random_state=0)[
    ["book", "entry_id", "date", "place", "ner_label", "matched_name", "region_historical"]
]

Random sample of resolved mentions for manual spot-checking


,book,entry_id,date,place,ner_label,matched_name,region_historical
7229,PintPL,ch11_e16,1915-06-16,Kapuskasing,GPE,"Kapuskasing, Cochrane District, Northeastern O...",Ontario
9205,WofW,ch05_e24,1812-09-24,Scott,ORG,"Scott, La Nouvelle-Beauce, Chaudière-Appalache...",Lower Canada
9431,WofW,ch07_e1,1812-11-01,Canada,GPE,Canada,Canada
4159,EftW,ch11_e10,1941-03-10,Diary,GPE,"Diary, Przasnysz, Przasnysz County, Masovian V...",Masovian Voivodeship
7353,SofS,chc02_e5,1847-05-08,Leary,GPE,"O'Leary, Town of O'Leary, Prince County, Princ...",Colony of Prince Edward Island
5711,OatmyD,ch01_e1,1897-05-24,Meg,ORG,"Meg, Queen Street West, West Queen West, Spadi...",Ontario
966,BFfH,ch07_e7,1916-06-15,Uxbridge,GPE,"Uxbridge, Durham Region, Ontario, Canada",Ontario
3555,DofmyC,chc04_e19,1759-07-31,Heights,GPE,"Heights, Martinborough Community, South Wairar...",Wellington
8389,TFN,ch01_e4,1912-05-04,Mill House,ORG,"Mill House, Harold Road, Lynn Valley, District...",British Columbia
1677,BUoL,ch05_e1,1885-04-01,Armand,ORG,"Armand, Edward Avenue, Chelmsford, Greater Sud...",Ontario
